In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import itertools

spark = SparkSession.builder \
    .appName("TMDB Exploratory Data Analysis") \
    .getOrCreate()

In [2]:
movies_df = spark.read.parquet("../data/processed/tmdb_movies_clean.parquet")
movies_df.cache()  # Cache karena akan dipakai beberapa kali
movies_df.show(5, truncate=False)

+------------------------+----+-----------+--------+-------------------+-----------+-----+--------------------+------------------------------------------------------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+----------+-------+---------+
|Movie Name              |Year|Certificate|Duration|Genre              |TMDB Rating|Votes|Director            |Stars                                                       |Grossed in $|Plot                                                                                                                                                                                         

## Statistika Deskriptif

In [24]:
movies_df.select(
    "Year", "Duration", "TMDB Rating", "Votes", "Grossed in $", "Popularity"
).describe().show()

+-------+------------------+------------------+------------------+------------------+-------------------+------------------+
|summary|              Year|          Duration|       TMDB Rating|             Votes|       Grossed in $|        Popularity|
+-------+------------------+------------------+------------------+------------------+-------------------+------------------+
|  count|              9740|              9740|              9740|              9740|               9740|              9740|
|   mean|2004.5041067761806|106.30369609856263| 6.731805338809012| 2101.797022587269|7.663537748131417E7|5.6489423408624315|
| stddev| 17.89469268409583| 23.13781367145706|0.6562514701588585|3360.4352092757545|1.626677457755464E8| 8.961339820259774|
|    min|              1902|                 2|             5.447|               300|                3.0|            0.0086|
|    max|              2026|               367|             8.716|             39092|      2.923706026E9|          699.9604|


Dataset ini mencakup 9.740 film yang diproduksi dalam rentang waktu yang sangat panjang, mulai dari tahun 1902 hingga 2026. Rata-rata tahun rilis berada di sekitar 2004, yang menunjukkan bahwa mayoritas film dalam dataset merupakan produksi era modern. Dari sisi durasi, film-film ini rata-rata berdurasi sekitar 106 menit atau sekitar 1 jam 46 menit, yang memang menjadi standar umum film bioskop. Meskipun demikian, terdapat variasi yang cukup besar, ada film yang hanya berdurasi 2 menit (kemungkinan film pendek) hingga yang terpanjang mencapai 367 menit atau hampir 6 jam.


Dari sisi kualitas, rata-rata rating TMDB berada di angka 6.73 dengan standar deviasi yang kecil yaitu 0.66. Ini berarti sebagian besar film berkumpul di kisaran rating 6 hingga 7.5, dan dataset ini cenderung hanya memuat film-film dengan kualitas di atas rata-rata karena tidak ditemukan film dengan rating yang sangat buruk, nilai terendah hanya 5.447 sementara tertinggi 8.716 milik The Shawshank Redemption.


Jumlah votes rata-rata hanya sekitar 2.102, namun standar deviasinya sangat tinggi mencapai 3.360. Ini menandakan bahwa distribusi votes sangat tidak merata, sebagian kecil film blockbuster mendapat puluhan ribu votes hingga maksimum 39.092, sementara mayoritas film hanya mendapat sedikit perhatian dari penonton.


Ketimpangan yang serupa juga terlihat pada pendapatan box office. Rata-rata pendapatan memang terlihat cukup besar di angka $76,6 juta, namun standar deviasinya mencapai $162,7 juta atau lebih dari dua kali lipat rata-ratanya. Hal ini menunjukkan bahwa hanya segelintir film yang benar-benar mendominasi pendapatan, film terlaris meraup hingga $2,92 miliar, sedangkan ada film yang hampir tidak menghasilkan uang sama sekali dengan pendapatan minimum hanya $3. Pola yang sama juga terlihat pada tingkat popularitas, di mana rata-ratanya hanya 5.65 namun nilai maksimumnya mencapai 699.96, jauh melampaui rata-rata. Ini mencerminkan efek "superstar" yang umum terjadi di industri hiburan, di mana hanya sedikit film yang benar-benar viral dan mendominasi perhatian publik, sementara sebagian besar film lainnya kurang dikenal luas.

# Korelasi Numerik (Sederhana)

In [25]:
numeric_cols = ["TMDB Rating", "Votes", "Grossed in $", "Duration", "Popularity", "Year"]
 
for i, col1 in enumerate(numeric_cols):
    for col2 in numeric_cols[i+1:]:
        corr_val = movies_df.stat.corr(col1, col2)
        print(f"  Korelasi {col1:20} ↔ {col2:20} : {corr_val:+.4f}")

  Korelasi TMDB Rating          ↔ Votes                : +0.2838
  Korelasi TMDB Rating          ↔ Grossed in $         : +0.1241
  Korelasi TMDB Rating          ↔ Duration             : +0.2236
  Korelasi TMDB Rating          ↔ Popularity           : +0.1205
  Korelasi TMDB Rating          ↔ Year                 : -0.2326
  Korelasi Votes                ↔ Grossed in $         : +0.7125
  Korelasi Votes                ↔ Duration             : +0.2409
  Korelasi Votes                ↔ Popularity           : +0.2534
  Korelasi Votes                ↔ Year                 : +0.0665
  Korelasi Grossed in $         ↔ Duration             : +0.2179
  Korelasi Grossed in $         ↔ Popularity           : +0.2450
  Korelasi Grossed in $         ↔ Year                 : +0.0937
  Korelasi Duration             ↔ Popularity           : +0.1174
  Korelasi Duration             ↔ Year                 : -0.0029
  Korelasi Popularity           ↔ Year                 : +0.0604


Temuan paling mencolok dari analisis korelasi ini adalah hubungan yang sangat kuat antara Votes dan Grossed in $ dengan nilai korelasi +0.71. Ini masuk akal secara logis, film yang ditonton banyak orang di bioskop cenderung juga banyak diulas dan diberi penilaian, sehingga jumlah votes dan pendapatan box office bergerak searah secara signifikan. Ini adalah satu-satunya pasangan variabel yang memiliki hubungan kuat dalam dataset ini.


Sementara itu, TMDB Rating ternyata hanya memiliki hubungan yang lemah terhadap semua variabel lainnya. Korelasi rating dengan votes (+0.28), durasi (+0.22), pendapatan (+0.12), dan popularitas (+0.12) semuanya tergolong rendah. Artinya, sebuah film yang mendapat rating tinggi belum tentu menghasilkan pendapatan besar, ditonton banyak orang, atau menjadi viral. Kualitas film menurut pengguna TMDB tampaknya tidak terlalu menentukan kesuksesan komersialnya.


Yang menarik, korelasi antara Rating dan Tahun Rilis bernilai negatif (-0.23). Ini mengindikasikan bahwa film-film lama cenderung mendapat rating lebih tinggi dibanding film baru. Kemungkinan besar karena film klasik yang bertahan hingga kini adalah yang memang benar-benar berkualitas dan telah teruji oleh waktu, sementara film modern masih bercampur antara yang bagus dan yang biasa-biasa saja.


Variabel durasi film juga menunjukkan pola yang menarik, korelasinya positif terhadap rating (+0.22), votes (+0.24), dan pendapatan (+0.22), meski tetap dalam kategori lemah. Ini memberi petunjuk bahwa film yang lebih panjang sedikit cenderung lebih serius dan ambisius, sehingga mendapat respons yang lebih baik dari penonton. Sedangkan popularitas memiliki korelasi yang sangat rendah terhadap hampir semua variabel, menunjukkan bahwa popularitas di TMDB lebih bersifat sesaat dan tidak selalu mencerminkan kualitas maupun kesuksesan komersial sebuah film.

# Analisis Film

## Distribusi Film per Dekade

In [26]:
movies_df.withColumn(
    "Decade", (F.col("Year") / 10).cast("int") * 10
).groupBy("Decade") \
 .agg(F.count("*").alias("Jumlah_Film")) \
 .orderBy("Decade") \
 .show()

+------+-----------+
|Decade|Jumlah_Film|
+------+-----------+
|  1900|          2|
|  1910|          4|
|  1920|         26|
|  1930|         48|
|  1940|         83|
|  1950|        152|
|  1960|        249|
|  1970|        359|
|  1980|        696|
|  1990|       1125|
|  2000|       2059|
|  2010|       3378|
|  2020|       1559|
+------+-----------+



## Golden Age of Cinema

In [27]:
movies_df.withColumn("Era",
    F.when(F.col("Year") < 1960, "Classic (<1960)")
     .when(F.col("Year") < 1980, "New Hollywood (1960–79)")
     .when(F.col("Year") < 2000, "Blockbuster Era (1980–99)")
     .when(F.col("Year") < 2015, "Digital Age (2000–14)")
     .otherwise("Modern (2015+)")
).groupBy("Era").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"),    3).alias("Avg Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
    F.round(F.avg("Votes"),          0).alias("Avg Votes"),
).orderBy("Era").show(truncate=30)

+-------------------------+----------+----------+----------------------+---------+
|                      Era|Film Count|Avg Rating|Avg Revenue (Juta USD)|Avg Votes|
+-------------------------+----------+----------+----------------------+---------+
|Blockbuster Era (1980–99)|      1821|     6.755|                 67.63|   1893.0|
|          Classic (<1960)|       315|     7.457|                  20.1|   1151.0|
|    Digital Age (2000–14)|      3597|     6.601|                 88.66|   2526.0|
|           Modern (2015+)|      3399|     6.711|                 81.21|   1995.0|
|  New Hollywood (1960–79)|       608|     7.173|                 36.18|   1304.0|
+-------------------------+----------+----------+----------------------+---------+



Film-film klasik sebelum tahun 1960 memiliki rata-rata rating tertinggi sebesar 7.46, jauh di atas era-era lainnya. Namun jumlah filmnya paling sedikit, hanya 315 film, dan rata-rata pendapatannya juga paling rendah di angka $20,1 juta. Ini konsisten dengan temuan korelasi sebelumnya, film lama yang masih diingat hingga kini adalah yang benar-benar terbaik, sementara yang biasa-biasa saja sudah terlupakan oleh waktu.


Era New Hollywood (1960–1979) juga masih mempertahankan kualitas yang tinggi dengan rata-rata rating 7.17, meski pendapatan masih relatif kecil di $36,18 juta. Ini wajar mengingat industri film saat itu belum sebesar sekarang dan belum mengenal fenomena blockbuster global.


Memasuki Blockbuster Era (1980–1999), jumlah film melonjak menjadi 1.821 dengan rata-rata pendapatan $67,63 juta, menandakan industri film mulai berkembang pesat secara komersial. Namun rating rata-ratanya mulai turun ke 6.76, yang bisa diartikan bahwa produksi film yang semakin masif tidak selalu dibarengi dengan peningkatan kualitas.


Era Digital (2000–2014) adalah era dengan jumlah film terbanyak yaitu 3.597 film sekaligus rata-rata rating terendah sebesar 6.60, namun pendapatan rata-ratanya justru tertinggi di angka $88,66 juta. Ini menggambarkan era di mana industri film sangat berorientasi komersial, diproduksi dalam jumlah besar dan menghasilkan uang lebih banyak, tetapi tidak selalu memuaskan dari sisi kualitas.


Film Modern (2015 ke atas) sedikit membaik dari sisi rating (6.71) dibanding era digital, dengan pendapatan rata-rata $81,21 juta dan jumlah votes tertinggi kedua. Secara keseluruhan, tren yang terlihat jelas adalah semakin modern sebuah film, semakin besar pendapatannya namun semakin rendah ratingnya, mengonfirmasi bahwa kesuksesan komersial dan kualitas sinematik tidak selalu berjalan beriringan.

## Film Hidden Gems

In [28]:
print("Film Rating Tinggi, Popularity Rendah, Revenue Rendah")
med_pop  = movies_df.approxQuantile("Popularity",    [0.5], 0.05)[0]
med_rev  = movies_df.approxQuantile("Grossed in $",[0.5], 0.05)[0]
 
print(f"  Median Popularity      : {med_pop:.2f}")
print(f"  Median Revenue         : ${med_rev/1e6:.1f} Juta")
print()
 
movies_df.filter(
    (F.col("TMDB Rating") >= 8.0) &
    (F.col("Popularity")  <= med_pop) &
    (F.col("Grossed in $") <= med_rev)
).select("Movie Name", "Year", "TMDB Rating", "Popularity",
         F.round(F.col("Grossed in $")/1e6, 2).alias("Revenue (Juta USD)")) \
 .orderBy(F.desc("TMDB Rating")) \
 .show(truncate=35)

Film Rating Tinggi, Popularity Rendah, Revenue Rendah
  Median Popularity      : 4.43
  Median Revenue         : $28.6 Juta

+-----------------------------------+----+-----------+----------+------------------+
|                         Movie Name|Year|TMDB Rating|Popularity|Revenue (Juta USD)|
+-----------------------------------+----+-----------+----------+------------------+
|             Grave of the Fireflies|1988|      8.444|     0.012|              0.84|
|                       A Dog's Will|2000|       8.42|    3.7039|               4.9|
|                  Impossible Things|2021|      8.415|    2.5331|              28.6|
|                  Gabriel's Inferno|2020|      8.393|    2.9405|              28.6|
|        Gabriel's Inferno: Part III|2020|      8.362|    2.9918|              28.6|
|                 Dedicated to my ex|2019|        8.3|    1.9642|              1.32|
|    We All Loved Each Other So Much|1974|      8.289|    2.1459|              28.6|
|                 Hotarub

Semua film hidden gem ini memiliki tingkat popularitas di bawah median 4.43 dan pendapatan di bawah median $28,6 juta, padahal kualitasnya bersaing dengan film-film blockbuster terkenal.


Yang paling menonjol adalah Grave of the Fireflies (1988), sebuah film animasi Jepang yang meraih rating tertinggi dalam daftar ini yaitu 8.44, namun popularitasnya hanya 0.012 nyaris nol, dengan pendapatan hanya $840,000. Ini menunjukkan betapa film ini sangat tidak dikenal secara luas padahal kualitasnya luar biasa. Hal serupa juga terlihat pada Hotarubi no Mori e (2011) yang popularitasnya bahkan lebih rendah lagi di angka 0.0086, terendah dalam seluruh daftar.


Menariknya, banyak film dalam daftar ini berasal dari sinema internasional non-Hollywood, seperti film Jepang, Italia, Meksiko, hingga Prancis. Tokyo Story (1953) dan The Seventh Seal (1957) misalnya, adalah mahakarya sinema dunia yang sudah diakui para kritikus selama puluhan tahun, namun tetap tidak populer di kalangan penonton umum. Ini mengonfirmasi bahwa film berbahasa asing dan film art-house cenderung tersaring dari radar penonton mainstream meski kualitasnya tidak kalah.


Beberapa film seperti Gabriel's Inferno dan variannya, serta beberapa judul lain, menampilkan nilai revenue tepat di angka $28,6 juta yang merupakan nilai median, kemungkinan besar ini adalah nilai default yang diisi otomatis karena data revenue aslinya tidak tersedia, sehingga perlu diinterpretasikan dengan hati-hati.


Secara keseluruhan, daftar ini membuktikan bahwa rating tinggi tidak menjamin popularitas atau kesuksesan komersial. Faktor seperti bahasa, distribusi terbatas, genre yang tidak mainstream, hingga kurangnya promosi bisa membuat sebuah film berkualitas tetap tersembunyi dari jutaan penonton yang justru mungkin akan menyukainya.

# Analisis Rating

In [29]:
movies_df.agg(
    F.round(F.min("TMDB Rating"),  3).alias("Rating Minimum"),
    F.round(F.max("TMDB Rating"),  3).alias("Rating Maksimum"),
    F.round(F.avg("TMDB Rating"),  3).alias("Rating Rata-rata"),
    F.round(F.stddev("TMDB Rating"), 3).alias("Standar Deviasi"),
).show()

+--------------+---------------+----------------+---------------+
|Rating Minimum|Rating Maksimum|Rating Rata-rata|Standar Deviasi|
+--------------+---------------+----------------+---------------+
|         5.447|          8.716|           6.732|          0.656|
+--------------+---------------+----------------+---------------+



Rating film dalam dataset ini berkisar antara 5.447 hingga 8.716, dengan rata-rata sebesar 6.732. Artinya, secara umum film-film yang masuk dalam dataset ini sudah tergolong cukup baik, tidak ada film dengan rating sangat buruk seperti di bawah 5, yang mengindikasikan bahwa dataset ini memang dikurasi atau memiliki threshold kualitas minimum tertentu. Standar deviasi yang kecil yaitu 0.656 menunjukkan bahwa sebagian besar film berkumpul cukup rapat di sekitar nilai rata-rata.

In [30]:
# Kategori rating
print("  Distribusi Kategori Rating:")
movies_df.withColumn("Rating Category",
    F.when(F.col("TMDB Rating") >= 8.5, "Masterpiece (≥8.5)")
     .when(F.col("TMDB Rating") >= 7.5, "Excellent (7.5–8.4)")
     .when(F.col("TMDB Rating") >= 6.5, "Good (6.5–7.4)")
     .otherwise("Below Average (<6.5)")
).groupBy("Rating Category") \
 .agg(F.count("*").alias("Jumlah Film")) \
 .orderBy(F.desc("Jumlah Film")) \
 .show()

  Distribusi Kategori Rating:
+--------------------+-----------+
|     Rating Category|Jumlah Film|
+--------------------+-----------+
|      Good (6.5–7.4)|       4667|
|Below Average (<6.5)|       3748|
| Excellent (7.5–8.4)|       1313|
|  Masterpiece (≥8.5)|         12|
+--------------------+-----------+



Dari total 9.740 film dalam dataset, kategori Good (6.5–7.4) mendominasi dengan 4.667 film atau sekitar 48% dari keseluruhan dataset. Ini sejalan dengan rata-rata rating 6.732 yang ditemukan sebelumnya, sebagian besar film memang berkumpul di rentang "cukup baik" namun belum luar biasa.


Kategori Below Average (di bawah 6.5) cukup besar dengan 3.748 film atau sekitar 38%. Jika digabung dengan kategori Good, maka hampir 86% film dalam dataset berada di bawah rating 7.5.


Kategori Excellent (7.5–8.4) hanya dihuni oleh 1.313 film atau sekitar 13%, sementara kategori Masterpiece (8.5 ke atas) sangat eksklusif dengan hanya 12 film dari hampir 10 ribu judul, kurang dari 0.2% dari total dataset. Ini menegaskan bahwa film yang benar-benar luar biasa sangatlah langka.

## Top Film Berdasarkan Rating

In [31]:
movies_df.select("Movie Name", "Year", "TMDB Rating", "Votes", "Director") \
         .orderBy(F.desc("TMDB Rating")) \
         .limit(10) \
         .show(truncate=40)

+---------------------------+----+-----------+-----+--------------------+
|                 Movie Name|Year|TMDB Rating|Votes|            Director|
+---------------------------+----+-----------+-----+--------------------+
|   The Shawshank Redemption|1994|      8.716|29897|      Frank Darabont|
|              The Godfather|1972|      8.688|22579|Francis Ford Coppola|
|      The Godfather Part II|1974|      8.572|13674|Francis Ford Coppola|
|           Schindler's List|1993|      8.566|17195|    Steven Spielberg|
|               12 Angry Men|1957|      8.555| 9809|        Sidney Lumet|
|      ¿Quieres ser mi hijo?|2023|      8.543|  301|       Ihtzi Hurtado|
|              Spirited Away|2001|      8.534|18050|      Hayao Miyazaki|
|            The Dark Knight|2008|      8.527|35314|   Christopher Nolan|
|Dilwale Dulhania Le Jayenge|1995|      8.517| 4565|       Aditya Chopra|
|             The Green Mile|1999|      8.504|18941|      Frank Darabont|
+---------------------------+----+----

Daftar 10 besar film dengan rating tertinggi ini memperlihatkan beberapa pola yang menarik. The Shawshank Redemption (1994) menempati posisi puncak dengan rating 8.716 sekaligus memiliki votes terbanyak kedua sebesar 29.897, menunjukkan bahwa film ini tidak hanya dicintai secara kritis tetapi juga oleh penonton luas. Posisi kedua dan ketiga ditempati oleh The Godfather dan The Godfather Part II, keduanya karya Francis Ford Coppola, menjadikannya satu-satunya sutradara yang memiliki dua film sekaligus dalam daftar 10 besar ini.


Yang menarik, Frank Darabont juga muncul dua kali dengan Schindler's List dan The Green Mile, mengonfirmasi bahwa sutradara berbakat cenderung konsisten menghasilkan karya terbaik. Dari sisi era, sebagian besar film dalam daftar ini berasal dari dekade 1990-an, yang tampaknya merupakan masa keemasan sinema berkualitas tinggi berdasarkan dataset ini.


Satu anomali yang mencolok adalah ¿Quieres ser mi hijo? (2023) di posisi keenam dengan rating 8.543, namun hanya memiliki 301 votes, jauh di bawah film-film lainnya yang rata-rata memiliki belasan hingga puluhan ribu votes. Ini patut dicermati karena rating tinggi dengan jumlah votes yang sangat sedikit bisa kurang representatif dan berpotensi bias, berbeda dengan The Dark Knight misalnya yang ratingnya sedikit lebih rendah namun didukung oleh 35.314 votes, terbanyak dalam daftar ini.


Kehadiran Spirited Away dari Jepang dan Dilwale Dulhania Le Jayenge dari India juga membuktikan bahwa film berkualitas terbaik tidak hanya datang dari Hollywood, melainkan dari berbagai penjuru dunia.

# Analisis Pendapatan (Box Office)

In [32]:
movies_df.agg(
    F.round(F.sum("Grossed in $") / 1e9, 3).alias("Total Pendapatan (Milyar $)"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Rata-rata (Juta $)"),
    F.round(F.max("Grossed in $") / 1e6, 2).alias("Pendapatan Tertinggi (Juta $)"),
    F.round(F.min("Grossed in $") / 1e6, 2).alias("Pendapatan Terendah (Juta $)"),
).show()

+---------------------------+------------------+-----------------------------+----------------------------+
|Total Pendapatan (Milyar $)|Rata-rata (Juta $)|Pendapatan Tertinggi (Juta $)|Pendapatan Terendah (Juta $)|
+---------------------------+------------------+-----------------------------+----------------------------+
|                    746.429|             76.64|                      2923.71|                         0.0|
+---------------------------+------------------+-----------------------------+----------------------------+



Total pendapatan seluruh film dalam dataset mencapai $746,4 miliar, angka yang sangat besar namun perlu dipahami konteksnya secara lebih dalam. Rata-rata pendapatan per film adalah $76,64 juta, namun seperti yang sudah terlihat dari analisis sebelumnya, angka rata-rata ini sangat menyesatkan karena distribusi pendapatan yang sangat tidak merata.


Kesenjangan antara film terlaris dan film dengan pendapatan terendah sangat ekstrem. Film terlaris berhasil meraup $2,92 miliar, hampir 38 kali lipat dari rata-rata, sementara ada film yang pendapatannya tercatat $0 merupakan film yang data revenuenya tidak tersedia dan diisi dengan nilai 0.

## Top 5 Film Terlaris

In [33]:
print("Top 5 Film Terlaris:")
movies_df.select(
    "Movie Name", "Year",
    F.round(F.col("Grossed in $") / 1e6, 2).alias("Pendapatan (Juta USD)")
).orderBy(F.desc("Grossed in $")) \
 .limit(5) \
 .show(truncate=40)

Top 5 Film Terlaris:
+------------------------+----+---------------------+
|              Movie Name|Year|Pendapatan (Juta USD)|
+------------------------+----+---------------------+
|                  Avatar|2009|              2923.71|
|       Avengers: Endgame|2019|              2799.44|
|Avatar: The Way of Water|2022|               2353.1|
|                 Titanic|1997|              2264.16|
|                Ne Zha 2|2025|              2259.82|
+------------------------+----+---------------------+



Avatar (2009) karya James Cameron masih bertahan di puncak dengan pendapatan $2,92 miliar, diikuti Avengers: Endgame (2019) di posisi kedua dengan $2,79 miliar. Keduanya adalah satu-satunya film yang menembus angka $2,5 miliar.
Menariknya, James Cameron hadir dua kali dalam daftar ini melalui Avatar dan Titanic (1997), menjadikannya sutradara paling dominan dalam hal pendapatan box office sepanjang sejarah. Sementara itu, Avatar: The Way of Water (2022) sebagai sekuel Avatar berhasil masuk posisi ketiga, membuktikan bahwa franchise Avatar tetap memiliki daya tarik komersial yang luar biasa meski dirilis 13 tahun setelah film pertamanya.


Yang paling mengejutkan adalah kehadiran Ne Zha 2 (2025) di posisi kelima dengan pendapatan $2,26 miliar. Ini adalah film animasi asal China yang membuktikan bahwa dominasi Hollywood di box office global mulai mendapat tantangan serius dari industri film Asia, khususnya China yang pasar domestiknya saja sudah mampu menghasilkan pendapatan setara film-film blockbuster Hollywood terbesar.

## Korelasi Fitur Numerik terhadap Revenue

In [34]:
print("Korelasi Semua Fitur Numerik terhadap Revenue:")
revenue_pairs = [
    ("TMDB Rating",  movies_df.stat.corr("TMDB Rating",  "Grossed in $")),
    ("Votes",        movies_df.stat.corr("Votes",         "Grossed in $")),
    ("Duration",     movies_df.stat.corr("Duration",      "Grossed in $")),
    ("Popularity",   movies_df.stat.corr("Popularity",    "Grossed in $")),
    ("Year",         movies_df.stat.corr("Year",          "Grossed in $")),
]
for feat, r in sorted(revenue_pairs, key=lambda x: -abs(x[1])):
    bar = "█" * int(abs(r) * 20)
    sign = "+" if r > 0 else "-"
    print(f"{feat:<18} {sign}{abs(r):.4f}  {bar}")

Korelasi Semua Fitur Numerik terhadap Revenue:
Votes              +0.7125  ██████████████
Popularity         +0.2450  ████
Duration           +0.2179  ████
TMDB Rating        +0.1241  ██
Year               +0.0937  █


Dari seluruh fitur numerik yang dianalisis, Votes adalah prediktor terkuat terhadap revenue dengan korelasi +0.71 yang tergolong kuat. Ini sangat masuk akal, film yang banyak ditonton orang di bioskop akan menghasilkan pendapatan besar sekaligus mendorong lebih banyak orang untuk memberikan penilaian, sehingga kedua variabel ini saling memperkuat satu sama lain. Dengan kata lain, votes bisa dijadikan indikator awal yang cukup andal untuk memperkirakan seberapa besar pendapatan sebuah film.


Sementara itu, Popularity, Duration, dan TMDB Rating hanya memiliki korelasi lemah terhadap revenue, masing-masing di angka +0.25, +0.22, dan +0.12. Artinya, seberapa viral sebuah film di platform TMDB, seberapa panjang durasinya, maupun seberapa tinggi ratingnya tidak terlalu menentukan apakah film tersebut akan menghasilkan uang banyak atau tidak. Ini kembali menegaskan temuan sebelumnya bahwa kualitas film dan kesuksesan komersialnya adalah dua hal yang berbeda.


Yang paling mengejutkan adalah Year (tahun rilis) memiliki korelasi paling lemah terhadap revenue dengan hanya +0.09. Ini berarti tahun rilis hampir tidak ada hubungannya dengan pendapatan.

## Rata-rata Revenue per Kategori Rating

In [35]:
movies_df.withColumn("Rating Tier",
    F.when(F.col("TMDB Rating") >= 8.5, "S-tier (≥8.5)")
     .when(F.col("TMDB Rating") >= 8.0, "A-tier (8.0–8.4)")
     .when(F.col("TMDB Rating") >= 7.0, "B-tier (7.0–7.9)")
     .otherwise("C-tier (<7.0)")
).groupBy("Rating Tier").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
    F.round(F.sum("Grossed in $") / 1e6, 2).alias("Total Revenue (Juta USD)"),
).orderBy(F.desc("Avg Revenue (Juta USD)")).show()

+----------------+----------+----------------------+------------------------+
|     Rating Tier|Film Count|Avg Revenue (Juta USD)|Total Revenue (Juta USD)|
+----------------+----------+----------------------+------------------------+
|   S-tier (≥8.5)|        12|                229.93|                 2759.21|
|A-tier (8.0–8.4)|       274|                152.19|                41698.94|
|B-tier (7.0–7.9)|      3195|                 91.25|               291532.87|
|   C-tier (<7.0)|      6259|                 65.58|               410437.55|
+----------------+----------+----------------------+------------------------+



Kualitas film berbanding lurus dengan rata-rata pendapatannya. Semakin tinggi rating tier sebuah film, semakin besar pula rata-rata pendapatan yang diperoleh. Film dengan S-tier (≥8.5) langka (hanya 12 film) ini mendulang rata-rata pendapatan tertinggi mencapai $229,93 juta. Film A-tier (8.0–8.4) juga menyumbang pendapatan rata-rata yang sangat besar, yakni $152,19 juta.


Sebaliknya, kualitas film yang biasa-biasa saja di C-tier (<7.0) hanya mampu menghasilkan pendapatan rata-rata terkecil sebesar $65,58 juta. Namun, karena distribusinya paling masif (mencapai 6.259 film), kategori ini justru menjadi penyumbang total pendapatan box office terbesar secara keseluruhan (lebih dari $410 miliar).

# Analisis Genre

In [36]:
# Explode genre ke baris terpisah
genre_df = movies_df.withColumn(
    "Genre_Split",
    F.explode(F.split(F.col("Genre"), ",\\s*"))
)

In [37]:
print("  Frekuensi Genre:")
genre_df.groupBy("Genre_Split") \
        .agg(
            F.count("*").alias("Jumlah Film"),
            F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
            F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Pendapatan (Juta USD)")
        ) \
        .orderBy(F.desc("Jumlah Film")) \
        .show()

  Frekuensi Genre:
+---------------+-----------+----------+-------------------------+
|    Genre_Split|Jumlah Film|Avg Rating|Avg Pendapatan (Juta USD)|
+---------------+-----------+----------+-------------------------+
|          Drama|       4533|     6.916|                    48.82|
|         Comedy|       3501|     6.613|                    71.85|
|       Thriller|       2591|      6.56|                    62.48|
|         Action|       2248|     6.623|                   132.23|
|        Romance|       1721|     6.769|                    60.12|
|      Adventure|       1602|     6.752|                    190.4|
|          Crime|       1564|     6.716|                    57.53|
|         Horror|       1311|     6.388|                    43.12|
|Science Fiction|       1152|      6.63|                   139.32|
|        Fantasy|       1096|     6.792|                   128.54|
|         Family|       1091|     6.791|                   134.92|
|      Animation|        912|     7.078|   

Drama mendominasi kuantitas dalam dataset dengan 4.533 judul film (hampir menyentuh setengah dataset). Meskipun sering dipuji dengan rating yang cukup tinggi secara rerata (6.91), sebagian besar film Drama berskala kecil dan secara komersial kurang bertenaga, hanya menghasilkan pendapatan rata-rata sebesar $48,82 juta, yang berada di bawah rata-rata umum industri. Tren yang sangat mirip juga terjadi pada genre populer lainnya seperti Comedy, Thriller, Romance, dan Crime; mereka banyak diproduksi masal namun pendapatan rata-ratanya cenderung moderat.

Di sisi lain, genre yang paling menguntungkan di industri justru dipimpin oleh Adventure dan Science Fiction. Adventure memiliki angka yang sangat istimewa dengan rata-rata $190,4 juta per judul film, disusul oleh Science Fiction ($139,32 juta), Family ($134,92 juta), dan Action ($132,23 juta). Genre-genre ini adalah tulang punggung dari "blockbuster event", karena menjanjikan tontonan visual sinematik epik dan hiburan skala besar, yang secara alami sukses mendatangkan penonton massal ke bioskop yang berujung pada pendapatan berlipat ganda.

Genre seperti War (7.08), Animation (7.07), dan History (7.06) konsisten meraih skor di atas 7. Ini mengindikasikan bahwa film-film bertemakan perang dan sejarah sering digarap serius untuk menyasar apresiasi penghargaan (awards), sementara film animasi terbukti sangat memuaskan penontonnya meskipun diproduksi dalam kuantitas yang relatif lebih sedikit. Pengecualian menarik terjadi pada Horror, yang meskipun rutin diproduksi (>1.300 film), namun merupakan genre dengan rekam jejak penilaian audiens terburuk, dengan rating paling rendah di angka 6.38.

## Tren Genre over Time

In [38]:
genre_decade = movies_df.withColumn(
    "Decade", (F.col("Year") / 10).cast("int") * 10
).withColumn(
    "Genre Single", F.explode(F.split(F.col("Genre"), ",\\s*"))
)

In [39]:
print("Jumlah Film per Genre per Dekade:")
genre_decade.groupBy("Decade", "Genre Single").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
).orderBy("Decade", F.desc("Film Count")).show(40, truncate=20)

Jumlah Film per Genre per Dekade:
+------+---------------+----------+----------+
|Decade|   Genre Single|Film Count|Avg Rating|
+------+---------------+----------+----------+
|  1900|      Adventure|         2|     7.448|
|  1900|        Western|         1|      6.99|
|  1900|        Fantasy|         1|     7.905|
|  1900|          Crime|         1|      6.99|
|  1900|         Action|         1|      6.99|
|  1900|Science Fiction|         1|     7.905|
|  1910|        History|         2|       6.5|
|  1910|          Drama|         2|       6.5|
|  1910|         Comedy|         2|      7.31|
|  1910|            War|         1|       5.9|
|  1910|        Romance|         1|       7.3|
|  1920|         Comedy|        13|     7.748|
|  1920|          Drama|        12|     7.843|
|  1920|        Romance|         9|     7.665|
|  1920|         Horror|         7|     7.595|
|  1920|            War|         3|     7.565|
|  1920|        Fantasy|         3|      7.78|
|  1920|         Action|  

Pada era paling awal sinema (dekade 1900-an hingga 1910-an), volume produksi film masih sangat eksklusif dan terbatas. Para sineas pionir saat itu sudah berani mengeksplorasi genre mendasar seperti Adventure, Comedy, dan Drama. Menariknya, di tengah segala keterbatasan setelan kamera dan efek visual pada zaman tersebut, film-film awal bergenre imajinatif seperti Fantasy dan Science Fiction masih mampu mencetak skor apresiasi klasik yang luar biasa tinggi (rata-rata rating hingga 7.9).


Lompat memasuki era industri yang lebih matang di dekade 1920-an hingga 1930-an, peta persaingan box office mulai dikuasai oleh Drama, Comedy, dan Romance. Ketiga genre ini terbukti menjadi tulang punggung primadona hiburan masyarakat klasik dengan jumlah volume produksi yang melonjak drastis (belasan hingga puluhan judul per genre). Dekade ini juga menjadi periode kelahiran genre penantang baru seperti Horror dan Crime yang secara meyakinkan mulai mendapat tempat khusus di hati penggemarnya.


Secara keseluruhan, analisis kualitas di rentang waktu awal ini menunjukkan betapa solidnya fondasi sinema yang dibangun. Hampir seluruh film-film dari dekade 1920-an dan 1930-an konsisten memiliki skor rating penonton di angka rata-rata solid di atas 7.1 hingga 7.8. Ini menegaskan status mereka sebagai sebuah mahakarya sinema klasik era "Golden Age" yang diabadikan oleh standar kualitas film yang tak lekang oleh waktu.

In [40]:
print("Genre Paling Populer per Dekade (Top 1):")
window_decade = Window.partitionBy("Decade").orderBy(F.desc("Film Count"))
genre_decade.groupBy("Decade", "Genre Single").agg(
    F.count("*").alias("Film Count"),
).withColumn("Rank", F.rank().over(window_decade)) \
 .filter(F.col("Rank") == 1) \
 .select("Decade", "Genre Single", "Film Count") \
 .orderBy("Decade") \
 .show()

Genre Paling Populer per Dekade (Top 1):
+------+------------+----------+
|Decade|Genre Single|Film Count|
+------+------------+----------+
|  1900|   Adventure|         2|
|  1910|       Drama|         2|
|  1910|     History|         2|
|  1910|      Comedy|         2|
|  1920|      Comedy|        13|
|  1930|       Drama|        23|
|  1940|       Drama|        48|
|  1950|       Drama|        89|
|  1960|       Drama|       137|
|  1970|       Drama|       169|
|  1980|      Comedy|       279|
|  1990|       Drama|       551|
|  2000|       Drama|       984|
|  2010|       Drama|      1646|
|  2020|       Drama|       626|
+------+------------+----------+



## Window Function - Ranking per Genre

In [41]:
genre_exploded = movies_df.withColumn(
    "Genre Single",
    F.explode(F.split(F.col("Genre"), ",\\s*"))
)
 
window_spec = Window.partitionBy("Genre Single").orderBy(F.desc("TMDB Rating"))
 
genre_exploded.withColumn("Rank", F.rank().over(window_spec)) \
              .filter(F.col("Rank") <= 3) \
              .select("Genre Single", "Rank", "Movie Name", "Year", "TMDB Rating") \
              .orderBy("Genre Single", "Rank") \
              .show(50, truncate=40)

+---------------+----+----------------------------------------+----+-----------+
|   Genre Single|Rank|                              Movie Name|Year|TMDB Rating|
+---------------+----+----------------------------------------+----+-----------+
|               |   1|                                  Return|1975|      6.921|
|         Action|   1|                         The Dark Knight|2008|      8.527|
|         Action|   2|                           Seven Samurai|1954|        8.5|
|         Action|   3|The Lord of the Rings: The Return of ...|2003|      8.494|
|      Adventure|   1|The Lord of the Rings: The Return of ...|2003|      8.494|
|      Adventure|   2|                            Interstellar|2014|      8.468|
|      Adventure|   3|The Lord of the Rings: The Fellowship...|2001|      8.428|
|      Animation|   1|                           Spirited Away|2001|      8.534|
|      Animation|   2|                              Your Name.|2016|      8.478|
|      Animation|   3|      

Dari hasil ranking film terbaik per genre ini, terlihat beberapa pola yang sangat menarik. Genre Crime dan Drama memiliki tiga besar yang identik persis, The Shawshank Redemption, The Godfather, dan The Godfather Part II, yang wajar karena ketiga film ini memang bergenre ganda Crime-Drama sekaligus. Ketiganya adalah film-film legendaris yang sudah teruji kualitasnya selama puluhan tahun dan hampir tidak terbantahkan posisinya sebagai karya terbaik dalam sejarah sinema.

Beberapa film tampil di lebih dari satu genre sekaligus, mencerminkan sifat multi-genre dari film-film berkualitas tinggi. Spirited Away misalnya mendominasi tiga genre sekaligus, Animation, Family, dan Fantasy, semuanya di posisi pertama, menjadikannya film paling dominan dalam analisis ini. Begitu pula The Lord of the Rings: The Return of the King yang muncul di genre Action, Adventure, dan Fantasy, serta The Dark Knight yang memuncaki dua genre sekaligus yaitu Action dan Thriller.

Genre Animation menampilkan dominasi kuat film-film Jepang dengan Spirited Away, Your Name., dan Grave of the Fireflies mengisi seluruh tiga besar, tidak ada satu pun film animasi Hollywood yang masuk. Ini adalah bukti nyata bahwa anime Jepang diakui memiliki kualitas sinematik yang luar biasa oleh pengguna TMDB secara global.

Genre Horror menarik untuk dicermati karena tiga besarnya semuanya adalah film lawas, Psycho (1960), The Shining (1980), dan Michael Jackson's Thriller (1983). Ini mengindikasikan bahwa film horor klasik masih dianggap lebih unggul kualitasnya dibanding produksi horor modern yang jumlahnya jauh lebih banyak.

Satu anomali yang perlu dicatat adalah munculnya ¿Quieres ser mi hijo? (2023) di puncak genre Comedy dan Romance dengan rating 8.543, padahal seperti yang sudah dibahas sebelumnya film ini hanya memiliki 301 votes. Posisinya di peringkat pertama perlu diinterpretasikan dengan hati-hati karena kurangnya jumlah penilai membuatnya kurang representatif dibanding film-film lain di daftar ini yang didukung oleh ribuan hingga puluhan ribu votes.

# Analisis Sutradara

## Sutradara dengan Film Terbanyak

In [42]:
print("Sutradara dengan Film Terbanyak:")
movies_df.groupBy("Director") \
         .agg(
             F.count("*").alias("Jumlah Film"),
             F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
             F.round(F.sum("Grossed in $") / 1e6, 2).alias("Total Pendapatan (Juta USD)")
         ) \
         .orderBy(F.desc("Jumlah Film")) \
         .limit(10) \
         .show(truncate=30)

Sutradara dengan Film Terbanyak:
+-----------------+-----------+----------+---------------------------+
|         Director|Jumlah Film|Avg Rating|Total Pendapatan (Juta USD)|
+-----------------+-----------+----------+---------------------------+
|      Woody Allen|         44|     6.764|                    1302.62|
|   Clint Eastwood|         36|     6.907|                    3460.87|
| Steven Spielberg|         33|     7.199|                   10637.39|
| Alfred Hitchcock|         30|     7.306|                     476.76|
|     Ridley Scott|         26|     6.877|                    4895.08|
|  Martin Scorsese|         25|      7.41|                    2390.49|
|Steven Soderbergh|         24|     6.513|                    2467.53|
|       Ron Howard|         22|      6.83|                    4159.32|
|       Tim Burton|         21|     6.971|                    4870.32|
|  Robert Zemeckis|         20|     7.023|                    4360.86|
+-----------------+-----------+----------+--

Dari 10 sutradara, Woody Allen menjadi yang paling produktif dengan 44 film, hampir dua kali lipat lebih banyak dari posisi terbawah. Namun dari sisi pendapatan, Woody Allen justru paling kecil di daftar ini hanya $1,3 miliar total, mencerminkan karakteristik filmografinya yang cenderung berskala kecil.

Di sini terdapat dua pola yang saling bertolak belakang. Alfred Hitchcock dan Martin Scorsese memimpin dalam hal kualitas dengan rata-rata rating tertinggi masing-masing 7.31 dan 7.41, meski bukan yang paling banyak memproduksi film. Sebaliknya, Steven Spielberg berhasil membuktikan diri sebagai satu-satunya sutradara yang unggul di kedua sisi sekaligus, rating tinggi (7.20) dan total pendapatan yang jauh melampaui semua saingannya dengan $10,6 miliar, lebih dari dua kali lipat posisi kedua Ridley Scott. Angka fantastis ini menegaskan posisi Spielberg sebagai sutradara tersukses secara komersial sekaligus paling konsisten kualitasnya dalam sejarah Hollywood.

## Sutradara dengan Filmografi paling Stabil Kualitasnya

In [43]:
director_stats = movies_df.groupBy("Director").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"),    3).alias("Avg Rating"),
    F.round(F.stddev("TMDB Rating"), 3).alias("Stddev Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
    F.round(F.sum("Grossed in $") / 1e6, 2).alias("Total Revenue (Juta USD)"),
)
director_stats.orderBy(F.desc("Film Count")).show(10, truncate=25)

+-----------------+----------+----------+-------------+----------------------+------------------------+
|         Director|Film Count|Avg Rating|Stddev Rating|Avg Revenue (Juta USD)|Total Revenue (Juta USD)|
+-----------------+----------+----------+-------------+----------------------+------------------------+
|      Woody Allen|        44|     6.764|        0.484|                  29.6|                 1302.62|
|   Clint Eastwood|        36|     6.907|        0.612|                 96.14|                 3460.87|
| Steven Spielberg|        33|     7.199|        0.637|                322.35|                10637.39|
| Alfred Hitchcock|        30|     7.306|        0.552|                 15.89|                  476.76|
|     Ridley Scott|        26|     6.877|        0.653|                188.27|                 4895.08|
|  Martin Scorsese|        25|      7.41|         0.54|                 95.62|                 2390.49|
|Steven Soderbergh|        24|     6.513|        0.453|         

Woody Allen menempati posisi teratas dengan 44 film, jauh lebih banyak dari sutradara lainnya. Namun rata-rata revenue hanya $29,6 juta per film. Ini sangat konsisten dengan gaya Woody Allen yang dikenal sebagai pembuat film art-house dan komedi intelektual dengan budget kecil, bukan blockbuster komersial.

Di sisi lain, Steven Spielberg menjadi sutradara paling dominan secara keseluruhan meski hanya memiliki 33 film. Ia mencatat rata-rata rating tertinggi kedua (7.199), rata-rata revenue tertinggi ($322,35 juta per film), dan total revenue tertinggi yang sangat mencolok yaitu $10,6 miliar, hampir tiga kali lipat dibanding Clint Eastwood di posisi kedua dengan $3,46 miliar.

Steven Soderbergh menjadi perhatian karena memiliki rata-rata rating terendah dalam daftar ini (6.513) sekaligus standar deviasi terkecil (0.453), artinya filmnya konsisten namun di level kualitas yang lebih rendah dibanding rekan-rekannya. Robert Zemeckis sebaliknya mencatat standar deviasi tertinggi (0.683), mengindikasikan kualitas filmnya paling tidak konsisten, ada yang sangat bagus seperti Forrest Gump, namun ada pula yang jauh di bawah ekspektasi.

# Analisis Aktor

In [44]:
# Ambil Top 50 Aktor
top_50 = movies_df.withColumn("Actor", F.explode("Stars")) \
    .groupBy("Actor").count().orderBy(F.desc("count")).limit(50).collect()
top_50_actors = [row["Actor"] for row in top_50]

In [45]:
# Buat df khusus aktor
actor_df = movies_df.withColumn("Actor", F.explode("Stars")) \
    .filter(F.col("Actor").isin(top_50_actors))

In [46]:
# Rata-rata Rating per Aktor
actor_rating = actor_df.groupBy("Actor").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
    F.round(F.min("TMDB Rating"), 3).alias("Min Rating"),
    F.round(F.max("TMDB Rating"), 3).alias("Max Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)")
)

## Aktor dengan Rating di Atas Rata-Rata Global

In [47]:
global_avg = movies_df.agg(F.avg("TMDB Rating")).first()[0]
print(f"Aktor yang Avg Rating-nya di Atas Global ({global_avg:.3f}):")
actor_rating.filter(
    (F.col("Film Count") >= 2) & (F.col("Avg Rating") > global_avg)
).orderBy(F.desc("Avg Rating")).show(10, truncate=30)

Aktor yang Avg Rating-nya di Atas Global (6.732):
+-----------------+----------+----------+----------+----------+----------------------+
|            Actor|Film Count|Avg Rating|Min Rating|Max Rating|Avg Revenue (Juta USD)|
+-----------------+----------+----------+----------+----------+----------------------+
|        Brad Pitt|        40|     7.018|     5.516|     8.438|                196.66|
|        Tom Hanks|        55|     6.972|     5.524|     8.504|                204.47|
|    Ralph Fiennes|        37|     6.932|     5.757|     8.566|                177.81|
|        Al Pacino|        36|     6.915|     5.592|     8.688|                 68.65|
|   Clint Eastwood|        43|     6.909|     5.763|     8.467|                  60.1|
|       Tom Cruise|        39|     6.909|     5.518|     8.155|                331.58|
|   Dustin Hoffman|        38|     6.877|     5.654|       7.8|                152.52|
|Denzel Washington|        41|     6.859|     5.503|       7.7|                 

Robert De Niro adalah aktor paling produktif dalam daftar ini dengan 72 film, jauh melampaui semua aktor lainnya. Namun menariknya, meski paling banyak bermain film, rata-rata ratingnya (6.832) justru salah satu yang terendah dalam daftar ini. Ini wajar karena semakin banyak film yang dibintangi, semakin besar kemungkinan ada beberapa film yang berkualitas rendah ikut masuk ke dalam perhitungan.

Tom Hanks dengan 55 film dan rata-rata rating 6.972 serta rata-rata revenue $204,47 juta adalah aktor yang paling berhasil menyeimbangkan kualitas dan komersialisme di antara semua aktor dalam daftar. Hampir setiap film yang dibintanginya tidak hanya mendapat respons positif dari penonton tetapi juga sukses secara finansial, menjadikannya salah satu aktor paling bankable sepanjang sejarah Hollywood.

Brad Pitt tampil sebagai aktor dengan rata-rata rating tertinggi dalam daftar yaitu 7.018, sekaligus memiliki rata-rata revenue yang besar di $196,66 juta. Ini menunjukkan bahwa Brad Pitt sangat selektif dalam memilih peran — dari 40 film yang dibintanginya, kualitasnya terjaga dengan baik dan mayoritas juga sukses secara komersial.

Yang paling mencolok dari sisi komersial adalah Tom Cruise dengan rata-rata revenue tertinggi dalam daftar yaitu $331,58 juta per film, hampir dua kali lipat Tom Hanks di posisi kedua. Meskipun rata-rata ratingnya tidak yang tertinggi (6.909), Tom Cruise adalah mesin pencetak uang yang paling andal, terutama melalui franchise Mission: Impossible yang terus menghasilkan pendapatan besar.

## Aktor dengan Film Terbanyak

In [48]:
actor_rating.orderBy(F.desc("Film Count")).show(10, truncate=30)

+-----------------+----------+----------+----------+----------+----------------------+
|            Actor|Film Count|Avg Rating|Min Rating|Max Rating|Avg Revenue (Juta USD)|
+-----------------+----------+----------+----------+----------+----------------------+
|   Robert De Niro|        72|     6.832|     5.533|     8.572|                 95.91|
|     Nicolas Cage|        61|     6.352|       5.5|     7.449|                 87.82|
|Samuel L. Jackson|        59|     6.535|     5.483|     8.486|                155.79|
|        Tom Hanks|        55|     6.972|     5.524|     8.504|                204.47|
|     Bruce Willis|        55|     6.474|       5.5|     8.486|                123.25|
|    Nicole Kidman|        49|     6.455|     5.454|      8.04|                 75.57|
|      Jackie Chan|        49|      6.66|     5.605|       7.5|                 78.21|
|   Morgan Freeman|        47|     6.613|     5.585|     8.716|                123.34|
|    Mark Wahlberg|        47|     6.624|  

## Pengaruh Aktor terhadap Revenue

In [49]:
actor_rating.orderBy(F.desc("Avg Revenue (Juta USD)")).show(10, truncate=30)

+------------------+----------+----------+----------+----------+----------------------+
|             Actor|Film Count|Avg Rating|Min Rating|Max Rating|Avg Revenue (Juta USD)|
+------------------+----------+----------+----------+----------+----------------------+
|        Tom Cruise|        39|     6.909|     5.518|     8.155|                331.58|
|     Harrison Ford|        40|     6.759|     5.558|     8.392|                259.46|
|       Johnny Depp|        43|     6.768|       5.6|      7.82|                 222.1|
|         Tom Hanks|        55|     6.972|     5.524|     8.504|                204.47|
|Scarlett Johansson|        34|     6.752|     5.625|     7.992|                 203.2|
|         Brad Pitt|        40|     7.018|     5.516|     8.438|                196.66|
|        Matt Damon|        40|     6.812|     5.848|     8.159|                 180.9|
|     Ralph Fiennes|        37|     6.932|     5.757|     8.566|                177.81|
|      Eddie Murphy|        39| 

Tom Cruise mempertahankan posisi teratas dengan rata-rata revenue $331,58 juta per film dari 39 film. Angka ini sangat jauh melampaui aktor lainnya.

Harrison Ford di posisi kedua dengan $259,46 juta per film dari 40 film juga menunjukkan daya tarik komersial yang luar biasa, sebagian besar didorong oleh franchise Star Wars dan Indiana Jones yang masing-masing menjadi fenomena budaya tersendiri. Sementara Johnny Depp di posisi ketiga dengan $222,1 juta rata-rata revenue sebagian besar ditopang oleh franchise Pirates of the Caribbean, karena tanpa franchise tersebut angkanya kemungkinan jauh lebih rendah.

Yang paling impresif adalah Brad Pitt yang berada di posisi keenam dengan $196,66 juta namun memiliki rata-rata rating tertinggi dalam daftar yaitu 7.018. Ini menjadikannya aktor dengan kombinasi kualitas dan komersialisme terbaik, filmnya tidak hanya laku keras tetapi juga berkualitas tinggi secara konsisten.

Scarlett Johansson adalah satu-satunya perempuan dalam daftar 10 besar ini dengan rata-rata revenue $203,2 juta dari 34 film, posisi yang sebagian besar didorong oleh perannya sebagai Black Widow dalam franchise Marvel Cinematic Universe.

## Pasangan Aktor Paling Sering Tampil Bersama

In [50]:
# Self-join actor_df untuk menemukan pasangan
a1 = actor_df.select(
    F.col("Movie Name"), F.col("Actor").alias("Actor A"), F.col("TMDB Rating")
)
a2 = actor_df.select(
    F.col("Movie Name"), F.col("Actor").alias("Actor B")
)
 
pair_df = a1.join(a2, on="Movie Name") \
            .filter(F.col("Actor A") < F.col("Actor B"))  # hindari duplikasi
 
print("Top 10 Pasangan Aktor (Film Bersama Terbanyak):")
pair_df.groupBy("Actor A", "Actor B").agg(
    F.count("*").alias("Films Together"),
    F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating Together"),
).orderBy(F.desc("Films Together"), F.desc("Avg Rating Together")) \
 .show(10, truncate=28)

Top 10 Pasangan Aktor (Film Bersama Terbanyak):
+----------------+-----------------+--------------+-------------------+
|         Actor A|          Actor B|Films Together|Avg Rating Together|
+----------------+-----------------+--------------+-------------------+
|       Al Pacino|   Robert De Niro|             4|              7.489|
|    Bruce Willis|Samuel L. Jackson|             4|              7.392|
|  Dustin Hoffman|   Robert De Niro|             4|              6.612|
|    Ben Kingsley|    Ralph Fiennes|             3|              7.559|
|     Ben Affleck|       Matt Damon|             3|              7.094|
|       Brad Pitt|       Matt Damon|             3|              6.915|
|   Nicole Kidman|       Tom Cruise|             3|              6.867|
|   Kevin Costner|  Tommy Lee Jones|             3|              6.781|
|   Ryan Reynolds|Samuel L. Jackson|             3|              6.603|
|Antonio Banderas|     Eddie Murphy|             3|              6.354|
+---------------

Pasangan Al Pacino dan Robert De Niro serta Bruce Willis dan Samuel L. Jackson sama-sama memegang rekor tampil bersama terbanyak dengan 4 film. Namun dari sisi kualitas, Al Pacino dan Robert De Niro unggul dengan rata-rata rating 7.489, tertinggi kedua dalam daftar. 

Ben Kingsley dan Ralph Fiennes menjadi pasangan dengan rata-rata rating tertinggi dalam daftar yaitu 7.559 meski hanya tampil bersama dalam 3 film. Keduanya adalah aktor watak kelas dunia yang dikenal sangat selektif dalam memilih peran, sehingga film-film yang mempertemukan mereka cenderung adalah proyek-proyek ambisius dan berkualitas tinggi seperti Schindler's List.

Pasangan Ben Affleck dan Matt Damon dengan rata-rata rating 7.094 mencerminkan persahabatan nyata keduanya di luar layar yang sudah terjalin sejak lama, keduanya bahkan pernah menulis skenario bersama untuk Good Will Hunting. Sementara Brad Pitt dan Matt Damon juga sering bertemu dalam proyek-proyek besar, terutama melalui franchise Ocean's Eleven yang mempertemukan banyak bintang sekaligus.

## Kolaborasi Director dan Actor

In [51]:
collab_df = actor_df.groupBy("Director", "Actor").agg(
    F.count("*").alias("Films Together"),
    F.round(F.avg("TMDB Rating"),    3).alias("Avg Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
)
 
print("Pasangan Direktur–Aktor Paling Sering Berkolaborasi:")
collab_df.filter(F.col("Films Together") >= 2) \
         .orderBy(F.desc("Films Together"), F.desc("Avg Rating")) \
         .show(10, truncate=30)

Pasangan Direktur–Aktor Paling Sering Berkolaborasi:
+---------------------+------------------+--------------+----------+----------------------+
|             Director|             Actor|Films Together|Avg Rating|Avg Revenue (Juta USD)|
+---------------------+------------------+--------------+----------+----------------------+
|       Clint Eastwood|    Clint Eastwood|            22|     6.845|                 82.24|
|      Martin Scorsese|    Robert De Niro|            10|     7.616|                 57.91|
|          Jackie Chan|       Jackie Chan|             8|     7.077|                 38.37|
|           Tim Burton|       Johnny Depp|             7|     7.049|                313.91|
|         Dennis Dugan|      Adam Sandler|             7|     6.314|                200.16|
|      Pedro Almodóvar|  Antonio Banderas|             6|     6.978|                 17.06|
|       Richard Donner|        Mel Gibson|             6|     6.854|                212.54|
|   Sylvester Stallone|Sylv

Hal pertama yang langsung mencolok adalah munculnya Clint Eastwood sebagai sutradara sekaligus aktor dalam filmnya sendiri sebanyak 22 kali. Ini mencerminkan kebiasaan Eastwood yang sepanjang karirnya sering merangkap peran di balik dan di depan kamera.

Dari sisi komersial, Tim Burton dan Johnny Depp adalah pasangan paling menguntungkan dengan rata-rata revenue $313,91 juta per film, tertinggi dalam daftar ini. Meskipun rata-rata ratingnya tidak istimewa di angka 7.049, kolaborasi keduanya terbukti sangat disukai penonton secara massal melalui film-film seperti Pirates of the Caribbean dan Alice in Wonderland.

Pasangan Dennis Dugan dan Adam Sandler dengan 7 film dan revenue rata-rata $200,16 juta, ratingnya adalah yang terendah kedua dalam daftar (6.314), namun pendapatannya sangat besar. Ini adalah contoh klasik film yang secara kritis dianggap biasa-biasa saja namun tetap laris di pasaran.

# Analisis Durasi

In [52]:
movies_df.agg(
    F.min("Duration").alias("Durasi Min (menit)"),
    F.max("Duration").alias("Durasi Max (menit)"),
    F.round(F.avg("Duration"), 1).alias("Durasi Rata-rata (menit)"),
).show()

+------------------+------------------+------------------------+
|Durasi Min (menit)|Durasi Max (menit)|Durasi Rata-rata (menit)|
+------------------+------------------+------------------------+
|                 2|               367|                   106.3|
+------------------+------------------+------------------------+



In [53]:
print("Kategori Durasi:")
movies_df.withColumn("Durasi Kategori",
    F.when(F.col("Duration") < 90,  "Pendek (<90 mnt)")
     .when(F.col("Duration") < 120, "Standar (90–119 mnt)")
     .when(F.col("Duration") < 150, "Panjang (120–149 mnt)")
     .otherwise("Epic (≥150 mnt)")
).groupBy("Durasi Kategori") \
 .agg(
     F.count("*").alias("Jumlah Film"),
     F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating")
 ) \
 .orderBy("Durasi Kategori") \
 .show()

Kategori Durasi:
+--------------------+-----------+----------+
|     Durasi Kategori|Jumlah Film|Avg Rating|
+--------------------+-----------+----------+
|     Epic (≥150 mnt)|        322|     7.376|
|Panjang (120–149 ...|       1801|     7.029|
|    Pendek (<90 mnt)|       1435|     6.667|
|Standar (90–119 mnt)|       6182|     6.627|
+--------------------+-----------+----------+



Tabel ini memperlihatkan hubungan semakin panjang sebuah film, semakin tinggi rata-rata ratingnya.

Film kategori Standar (90–119 menit) adalah yang paling banyak dengan 6.182 film atau sekitar 63% dari keseluruhan dataset, namun justru memiliki rata-rata rating terendah yaitu 6.627. Ini wajar karena durasi standar adalah pilihan paling umum dan paling mudah diproduksi, sehingga film-film dengan kualitas biasa-biasa saja pun masuk ke kategori ini dan menekan rata-rata ratingnya ke bawah.

Film Pendek (di bawah 90 menit) memiliki rata-rata rating 6.667 yang sedikit lebih tinggi dari kategori standar, kemungkinan karena banyak film pendek adalah karya-karya indie atau dokumenter yang dibuat dengan pendekatan lebih artistik meski skalanya kecil.

Film Panjang (120–149 menit) sudah menunjukkan peningkatan kualitas yang cukup signifikan dengan rata-rata rating 7.029, melampaui angka rata-rata global 6.732. Ini mengindikasikan bahwa sutradara yang berani memperpanjang filmnya di atas 2 jam biasanya memiliki cerita yang lebih kaya dan matang untuk diceritakan.

Puncaknya adalah film kategori Epic (150 menit ke atas) yang meskipun hanya berjumlah 322 film atau sekitar 3% dari dataset, memiliki rata-rata rating tertinggi yaitu 7.376. Film-film epic seperti The Godfather, Schindler's List, dan The Lord of the Rings memang cenderung adalah proyek-proyek ambisius yang digarap dengan sangat serius dari sisi cerita, produksi, maupun akting.

# Analisis Sertifikasi

In [54]:
movies_df.groupBy("Certificate") \
         .agg(
             F.count("*").alias("Jumlah Film"),
             F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating")
         ) \
         .orderBy(F.desc("Jumlah Film")) \
         .show()

+-----------+-----------+----------+
|Certificate|Jumlah Film|Avg Rating|
+-----------+-----------+----------+
|          R|       3807|     6.619|
|      PG-13|       2193|     6.665|
|    Unknown|       1301|     6.824|
|         PG|       1265|     6.832|
|         NR|        704|     7.068|
|          G|        425|     6.949|
|      NC-17|         45|     6.747|
+-----------+-----------+----------+



Distribusi sertifikasi film dalam dataset ini didominasi oleh film berrating R (Restricted) dengan 3.807 film atau sekitar 39% dari keseluruhan dataset, namun rata-rata ratingnya justru yang terendah yaitu 6.619. Ini menunjukkan bahwa film-film dewasa diproduksi dalam jumlah paling banyak namun tidak selalu menghasilkan kualitas yang lebih baik. Film PG-13 di posisi kedua dengan 2.193 film juga memiliki rata-rata rating yang hampir sama rendahnya di 6.665, mencerminkan bahwa kedua kategori ini banyak diisi oleh film-film mainstream yang lebih mengejar pasar luas daripada kualitas sinematik.

Yang paling menarik adalah film berrating NR (Not Rated) justru memiliki rata-rata rating tertinggi dalam daftar yaitu 7.068, jauh di atas kategori lainnya. Film-film NR umumnya adalah karya-karya independen, film festival, atau film asing yang tidak melalui proses sertifikasi resmi MPAA dan seperti yang sudah terlihat di analisis-analisis sebelumnya, film-film semacam ini cenderung lebih berorientasi artistik dan mendapat apresiasi lebih tinggi dari penonton yang menontonnya.

Film G (General Audiences) dengan rata-rata rating 6.949 juga tampil cukup baik, kemungkinan besar karena kategori ini banyak diisi oleh film animasi berkualitas tinggi seperti karya-karya Studio Ghibli dan Pixar yang sudah terbukti mendapat penilaian sangat positif dari penonton di semua usia.

# Analisis Bahasa

In [55]:
movies_df.groupBy("Original Language") \
         .agg(
             F.count("*").alias("Jumlah Film"),
             F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
             F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Pendapatan JutaUSD")
         ) \
         .orderBy(F.desc("Jumlah Film")) \
         .show()

+-----------------+-----------+----------+----------------------+
|Original Language|Jumlah Film|Avg Rating|Avg Pendapatan JutaUSD|
+-----------------+-----------+----------+----------------------+
|               en|       7584|      6.65|                 89.88|
|               fr|        637|     6.783|                 23.79|
|               it|        342|     6.886|                 22.21|
|               ja|        298|     7.359|                 38.03|
|               es|        194|     7.031|                 21.82|
|               de|        104|     7.051|                 23.15|
|               ko|         99|      7.33|                 35.56|
|               zh|         67|     7.124|                152.94|
|               cn|         65|     7.051|                 30.18|
|               ru|         53|     7.049|                 17.65|
|               sv|         40|     7.192|                 22.11|
|               hi|         40|     7.257|                 50.01|
|         

Film berbahasa Inggris (en) mendominasi dataset dengan 7.584 film atau sekitar 78% dari keseluruhan, dengan rata-rata revenue tertinggi yaitu $89,88 juta per film. Namun rata-rata ratingnya hanya 6.65, salah satu yang terendah dalam daftar. Ini sangat masuk akal mengingat Hollywood memproduksi film dalam jumlah sangat masif setiap tahunnya, sehingga banyak film berkualitas rendah ikut masuk dan menekan rata-rata ke bawah.

Film berbahasa Jepang (ja) tampil sebagai yang paling menonjol dari sisi kualitas dengan rata-rata rating 7.359 dari 298 film, tertinggi kedua dalam daftar. Ini konsisten dengan temuan sebelumnya di analisis genre Animation dan Hidden Gems, di mana film-film Jepang seperti Spirited Away, Your Name., dan Grave of the Fireflies selalu muncul di puncak.

Yang paling mengejutkan adalah film berbahasa Portugis (pt) memiliki rata-rata rating tertinggi dalam seluruh daftar yaitu 7.52 meski hanya dari 30 film, diikuti Turki (tr) dengan 7.47 dan Persia/Iran (fa) dengan 7.468. Ketiganya adalah sinema yang relatif niche di pasar global namun ternyata sangat diapresiasi oleh penonton yang menontonnya, kemungkinan besar karena film-film yang masuk ke platform internasional seperti TMDB adalah yang memang terbaik dari negaranya masing-masing.

Film berbahasa Korea (ko) dengan rata-rata rating 7.33 dari 99 film juga tampil sangat impresif, apalagi dengan rata-rata revenue $35,56 juta yang cukup besar untuk film non-Hollywood. Kebangkitan sinema Korea dalam satu dekade terakhir yang ditandai dengan kesuksesan Parasite di Oscar 2020.